In [1]:
import numpy as np
import pandas as pd

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
load = pd.read_csv(f"{path}/forecast_load.csv")
hpfc = pd.read_csv(f"{path}/hpfc.csv")
fut = pd.read_csv(f"{path}/futures_prices.csv")

In [3]:
a = np.log10(fut["market_activity"].to_numpy(dtype=float))
centres = np.array([a.min(), a.max()])
for _ in range(50):
    high = np.abs(a - centres[1]) <= np.abs(a - centres[0])
    centres = np.array([a[~high].mean(), a[high].mean()])
fut["eligible"] = high

In [4]:
grid = load[["timestamp_utc", "total_forecast_mwh"]].merge(hpfc[["timestamp_utc", "timestamp_local", "is_peak", "hpfc_eur_mwh"]], on="timestamp_utc")
L = grid["total_forecast_mwh"].to_numpy()
price = grid["hpfc_eur_mwh"].to_numpy()
peak = grid["is_peak"].astype(bool).to_numpy()
dates = grid["timestamp_local"].str[:10]

In [5]:
grid

,timestamp_utc,total_forecast_mwh,timestamp_local,is_peak,hpfc_eur_mwh
0,2023-12-31T23:00:00Z,1.077136,2024-01-01T00:00:00+01:00,0,54.346521
1,2023-12-31T23:15:00Z,1.003301,2024-01-01T00:15:00+01:00,0,54.346521
2,2023-12-31T23:30:00Z,0.933171,2024-01-01T00:30:00+01:00,0,54.346521
3,2023-12-31T23:45:00Z,0.865711,2024-01-01T00:45:00+01:00,0,54.346521
4,2024-01-01T00:00:00Z,0.800915,2024-01-01T01:00:00+01:00,0,52.553398
...,...,...,...,...,...
35131,2024-12-31T21:45:00Z,1.454299,2024-12-31T22:45:00+01:00,0,89.985670
35132,2024-12-31T22:00:00Z,1.395945,2024-12-31T23:00:00+01:00,0,78.845786
35133,2024-12-31T22:15:00Z,1.324609,2024-12-31T23:15:00+01:00,0,78.845786
35134,2024-12-31T22:30:00Z,1.246778,2024-12-31T23:30:00+01:00,0,78.845786


In [6]:
def run(name, tag, periods):
    sel = fut[fut["eligible"] & fut["delivery_period"].isin(periods)].reset_index(drop=True)

    # Energy delivered in each quarter-hour by a 1 MW position in each product
    A = np.zeros((len(grid), len(sel)))
    for k in range(len(sel)):
        m = (dates >= sel["delivery_start"][k][:10]) & (dates < sel["delivery_end_exclusive"][k][:10])
        m = m.to_numpy()
        if sel["load_type"][k] == "PEAK":
            m = m & peak
        A[:, k] = 0.25 * m  #0.25 per quarter-hour when active

    # One value-neutrality equation per non-overlapping delivery block
    C, d, block_names = [], [], []
    for p, blk in sel.groupby("delivery_period"):
        m = ((dates >= blk["delivery_start"].iloc[0][:10]) & (dates < blk["delivery_end_exclusive"].iloc[0][:10])).to_numpy()
        C.append(price[m] @ A[m])  #spot-market value of 1 mwh of each contract
        d.append(price[m] @ L[m])  #actual energy cost acc to forecast_load from t1(L)
        block_names.append(p)
    C, d = np.array(C), np.array(d)  #C*x=d where x = the amount of mwh needed

    # Least-squares fit under the equality constraints, forcing negative positions to zero
    n = A.shape[1]
    fixed = np.zeros(n, dtype=bool)
    for _ in range(n + 1):
        E = np.vstack([C, np.eye(n)[fixed]])
        f = np.concatenate([d, np.zeros(int(fixed.sum()))])
        K = np.block([[2 * A.T @ A, E.T], [E, np.zeros((len(f), len(f)))]])
        x = np.linalg.solve(K, np.concatenate([2 * A.T @ L, f]))[:n]
        if (x >= -1e-9).all():
            break
        fixed[np.argmin(x)] = True

    # Hedge energy profile and the requested checks
    H = A @ x
    print("\n", name)
    for k in range(n):
        print("  position", sel["delivery_period"][k], sel["load_type"][k], round(x[k], 4), "MW")  #contracts to buy to hedge for each quarter-hour
    print("  hedge notional energy:", round(H.sum(), 2), "MWh")  #total energy to be bought for the whole period
    print("  hedge value:", round(H @ price, 2), "EUR   forecast value:", round(L @ price, 2), "EUR")  #zero profit
    for i in range(len(block_names)):
        print("  value-neutrality error", block_names[i], round(C[i] @ x - d[i], 6), "EUR")
    print("  residual-load RMSE:", round(np.sqrt(np.mean((L - H) ** 2)), 6), "MWh")  #how good the regression of least squares worked
    print("  absolute residual Day-Ahead volume:", round(np.abs(L - H).sum(), 2), "MWh")  #sum of all daily-surplus and -shortage required to trade

    sel["position_mw"] = x
    sel["hedge_energy"] = x * A.sum(axis=0)
    sel[["delivery_period", "load_type", "position_mw", "hedge_energy", "price_eur_mwh"]].to_csv(f"{path}/task3_positions_"+tag+".csv", index=False)
    pd.DataFrame({"timestamp_utc": grid["timestamp_utc"], "hedge_energy": H, "futures_cost": A @ (x * sel["price_eur_mwh"].to_numpy())}).to_csv(f"{path}/task3_hedge_"+tag+".csv", index=False)

In [7]:
run("Coarse hedge (CAL)", "coarse", ["CAL"])
run("Granular hedge (M01-M03, Q2-Q4)", "granular", ["M01", "M02", "M03", "Q2", "Q3", "Q4"])


 Coarse hedge (CAL)
  position CAL BASE 4.3697 MW
  position CAL PEAK 1.88 MW
  hedge notional energy: 44294.41 MWh
  hedge value: 3477395.26 EUR   forecast value: 3477395.26 EUR
  value-neutrality error CAL 0.0 EUR
  residual-load RMSE: 0.441113 MWh
  absolute residual Day-Ahead volume: 12730.37 MWh

 Granular hedge (M01-M03, Q2-Q4)
  position Q2 BASE 4.019 MW
  position Q2 PEAK 1.6776 MW
  position Q3 BASE 3.776 MW
  position Q3 PEAK 1.5494 MW
  position Q4 BASE 4.6187 MW
  position Q4 PEAK 2.1116 MW
  position M01 BASE 5.0689 MW
  position M01 PEAK 2.3389 MW
  position M02 BASE 4.9758 MW
  position M02 PEAK 2.2302 MW
  position M03 BASE 4.6451 MW
  position M03 PEAK 1.97 MW
  hedge notional energy: 43915.13 MWh
  hedge value: 3477395.26 EUR   forecast value: 3477395.26 EUR
  value-neutrality error M01 0.0 EUR
  value-neutrality error M02 0.0 EUR
  value-neutrality error M03 -0.0 EUR
  value-neutrality error Q2 0.0 EUR
  value-neutrality error Q3 0.0 EUR
  value-neutrality error Q4 